In [0]:
%sql

-- *** Window Functions ***

-- Window Functions Synatx 
-- Window Function  [Partition Clause , Order Clause , Frame Clause]
-- AVG(Sales) OVER (PARTITION BY Region ORDER BY Sales ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT)


-- Perform Calclutions (Aggregations) on specifc subset of data without lossing the level of detials
-- Perform Calclutions across rows related to current row
-- Keep all rows (unlike Group by)
-- Analyzing while preserving all rows (details)
-- Window Functions are applied to a group of rows called a Window
-- Window is defined by PARTITION BY and ORDER BY
-- PARTITION BY defines the groups of rows
-- ORDER BY defines the order of rows within each group
-- Window Functions are applied to each row in the group


----------------------------------------------------------------------------------------------

-- Row_Number() is a Window Function
-- Assigns a unique number to each row within a group
-- The numbering starts at 1 for each group
-- The numbering is sequential and incremented by 1 for each row within the group
-- The numbering is reset to 1 for each new group


----------------------------------------------------------------------------------------------

--  RANK() :    SAME RANKING FOR TIES , GPAS AFTER 
--  DENSE_RANK() :  RANKING WITHOUT GAP FOR TIES 
-- LAG() :  PREVIOUS ROW VALUE
-- LEAD() :  NEXT ROW VALUE
-- NTILE() :  DIVIDE INTO GROUPS
-- CUME_DIST() :  CUMULATIVE PERCENTAGE
-- PERCENT_RANK()



/*

Window Expression Arguments
These categories define what goes inside the parentheses of the function before the OVER clause.

# Empty
Description: No arguments are needed because the function operates based solely on the row's position or the window definition.

Example: RANK() OVER (ORDER BY OrderDate)

# Column
Description: The function performs a calculation on a specific column/field.

Example: AVG(Sales) OVER (ORDER BY OrderDate)

# Number

Description: Certain functions require a specific numeric value to define how they operate (like buckets or offsets).

Example: NTILE(2) OVER (ORDER BY OrderDate) (Note: The image has a small typo "NTEIL", it is correctly spelled NTILE).

# Multiple Arguments

Description: Some advanced functions need several pieces of information (Column, Offset, and Default Value).

Example: LEAD(Sales, 2, 10) OVER (ORDER BY OrderDate)

# Conditional Logic

Description: You can wrap a CASE WHEN statement inside the function to only aggregate data that meets specific criteria.

Example: SUM(CASE WHEN Sales > 100 THEN 1 ELSE 0 END) OVER (ORDER BY OrderDate)
 
 */


/* 

Difference between Group By and Window Functions : 

1. GROUP BY Functions
These are Aggregate Functions used when you want to collapse multiple rows into a single summary row.

COUNT(expr): Counts the number of rows.

SUM(expr): Calculates the total sum of a numeric column.

AVG(expr): Calculates the average value.

MIN(expr): Finds the minimum value in the set.

MAX(expr): Finds the maximum value in the set.

2. WINDOW Functions
These functions allow you to perform calculations while preserving the individual rows. They are categorized into three main types:

A. Aggregate Functions (Used as Windows)
These are the same as the Group By functions, but when used with the OVER() clause, they don't collapse the rows.

COUNT(expr)

SUM(expr)

AVG(expr)

MIN(expr)

MAX(expr)

B. Rank Functions
These are used to assign a ranking or a number to each row within a partition.

ROW_NUMBER(): Assigns a unique sequential integer to rows.

RANK(): Assigns a rank with gaps (e.g., 1, 2, 2, 4).

DENSE_RANK(): Assigns a rank without gaps (e.g., 1, 2, 2, 3).

CUME_DIST(): Calculates the cumulative distribution of a value.

PERCENT_RANK(): Calculates the relative rank of a row as a percentage.

NTILE(n): Divides rows into n number of ranked groups (tiles).

C. Value (Analytics) Functions
These allow you to compare values across different rows in the same result set.

LEAD(expr, offset, default): Accesses data from a subsequent row (the "next" row).

LAG(expr, offset, default): Accesses data from a previous row (the "prior" row).

FIRST_VALUE(expr): Returns the value from the first row in the window.

LAST_VALUE(expr): Returns the value from the last row in the window. 

*/


In [0]:
%sql
-- 

select order_id,order_date, rank() over (order by order_date) as rank_num
 from orders


In [0]:
%sql

-- Show frist product bought by each customer

select * from 

(select c.customer_id, 
c.first_name, 
od.product_name, 
od.product_id,
row_number() over (partition by c.customer_id order by o.order_date) as Purchase_Sequence
from customers c
right join orders o on c.customer_id = o.customer_id
join order_details od on o.order_id = od.order_id  
)

where Purchase_Sequence= 1


In [0]:
%sql
select product_name , quantity ,
rank() over (order by quantity desc) as ranking
from order_details 


In [0]:
%sql
-- Assign bonus based on best-selling products in each order 

select  order_id,
 product_name,
quantity,
rank() over (partition by order_id  order by quantity desc) as ranking
from order_details

In [0]:
select 
product_name,
quantity,
dense_rank() over (order by quantity desc) as dens_rank,
rank() over (order by quantity desc) as rank


from order_details

In [0]:
-- lable products popularity tiers based on quntity

select
product_name ,
sum(quantity) as Total ,
dense_rank() over (order by sum(quantity) desc) as rank
from order_details
group by product_name




In [0]:
-- LAG ()   Compar with previous row
-- LEAD ()  compare with next row

select 
order_id,
order_date ,
LEAD(order_date) over (order by order_date) as Next_order_date 
from orders

/*
select 
order_id,
order_date ,
LAG(order_date) over (order by order_date) as previos_order_date 
from orders
*/



In [0]:
-- Analayz the customers trends #1 (period from last order)

select 
o.customer_id ,
o.order_id,
o.total_amount,
o.order_date,
LAG(o.order_date) over (partition by o.customer_id order by o.customer_id) as Previous_order_date
from orders o



In [0]:
-- Analayz the customers trends #1 (period from last order)
-- Analayz the customers trends #2 (did they buy more or less)

select 
o.customer_id ,
o.order_id,
o.order_date,
LAG(o.order_date) over (partition by o.customer_id order by o.customer_id) as Previous_order_date ,
o.total_amount,
LAG(o.total_amount) over (partition by o.customer_id order by o.customer_id) as previous_amount,
(o.total_amount-previous_amount) as difference
from orders o
-- order by order_date asc



In [0]:
-- SUM() --->   Running Total (qauntity) of Cummulative Sum

select order_id , product_name , quantity ,
sum(quantity) over (order by order_id) as total
from order_details
 

In [0]:
-- SUM() --->   Running Total (salary) of Cummulative Sum
select 
first_name,
department,
salary ,
sum(salary) over (partition by department order by emp_id) as running_total
from employees

/*
select 
first_name,
department,
salary ,
sum(salary) over (partition by department order by emp_id) as running_total
from employees
*/